<a href="https://colab.research.google.com/github/Kentakure/hazardmap/blob/main/Created-in-Google-Colab/Evacuation_Map_4546AroundMtSakurajima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests # zip ファイルをダウンロードするために追加
import zipfile  # zip ファイルを処理するために追加
import io       # メモリ内 zip ファイル処理用を開設
import tempfile # 一時ディレクトリを作成するために追加
import shutil   # 一時ディレクトリをクリーンアップするために追加
import os       # パス操作用に追加
import folium # マップ操作用に追加
from folium import plugins
import geopandas as gpd # GeoDataFrame 操作用に追加
import pandas as pd # DataFrame 操作用に追加
from geopy.geocoders import Nominatim #ジオコーディング用に追加
from datetime import datetime
from google.colab import files
import subprocess
import re # 破損XMLを正規表現でパースするために追加

# パラメータ群
# 地図の中心
DEFAULT_CENTER_OF_MAP = [31.562612, 130.722941] #https://geoshape.ex.nii.ac.jp/ka/resource/46/462016550.html
# 都道府県コード
PREF_CODE = ['45000','46000']
# GSIデータURL
GSI_URL = 'https://japanhazardmap.github.io/GISDataOriginalZips/MLITGSI/DesignatedEvacuationSitesandShelters/'
# P20のZIPファイル名
P20_ZIP_NAME = ['P20-12_45','P20-12_46']
# KSJデータURL
KSJ_URL = 'https://japanhazardmap.github.io/GISDataOriginalZips/MLITKSJ/P20'
# 欠損した.shxファイルを自動で修復・再生成してShapefileをロード可能にする環境変数を設定
os.environ["SHAPE_RESTORE_SHX"] = "YES"

# geopy を使用して住所入力から座標を取得します
print("地図の中央に配置する住所を英語またはローマ字で入力して下さい。")
print("（例:「Tokyo Station」）また空白の場合はデフォルトの座標を使用します。")
address_input = input("決定後Enterキーを押して下さい：")
if address_input:
    geolocator = Nominatim(user_agent="geoapi_colab_map")
    try:
        location = geolocator.geocode(address_input)
        if location:
            center = [location.latitude, location.longitude]
            print(f"地図の中心: {location.address} ({center[0]}, {center[1]}) ")
        else:
            print(f"デフォルトの中心を使用しているため、'{address_input}'の座標が見つかりませんでした。")
    except Exception as e:
        print(f"ジオコーディングエラー '{address_input}': {e} デフォルトの中心を使用。")

# 'center' が定義されていない場合 (たとえば、ユーザーが入力セルをスキップした場合)、堅牢性のためにデフォルトが使用されます。
if 'center' not in locals():
    print("「警告: 'center' 変数が見つかりません。デフォルトの座標を使用します。」")
    center = DEFAULT_CENTER_OF_MAP

# OpenStreetMap
fmap1 = folium.Map(location=center,
                   zoom_start = 15,
                   #zoom_control=False, # ズームボタンを表示しない
                   scrollWheelZoom=False, # マウスホイールでのズームを無効化
                   doubleClickZoom=False, # ダブルクリックでのズームを無効化
                   #dragging=False # ドラッグでの移動を無効化
)

# 値「1」を「該当」に置き換え、「0」または空の値を「非該当」に置き換えるヘルパー関数
def _replace_categorical_values(gdf, columns):
    for col in columns:
        if col in gdf.columns:
            # 最初に数値に変換し、エラーを NaN に強制してから、NaN を 0 で埋めます
            # これは空の文字列、数値以外の値を「非該当」として処理します
            gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0).astype(int)
            gdf[col] = gdf[col].map({1: '該当', 0: '×非該当×'})
            # 変換後も値が 0 または 1 にならなかった場合は、残った NaN を埋めます
            gdf[col] = gdf[col].fillna('×非該当×')
    return gdf

# CSV を GeoDataFrame に変換するヘルパー関数
def csv_to_gdf(df_in, suffix_key_for_log):
    df = df_in.copy() # Work on a copy
    lat_cols = ['緯度', 'latitude', 'lat', 'Y']
    lon_cols = ['経度', 'longitude', 'lon', 'X']

    found_lat_col = None
    found_lon_col = None

    # 大文字と小文字を区別せずに一致する列を検索します
    for col in df.columns:
        if any(lc.lower() == col.lower() for lc in lat_cols):
            found_lat_col = col
        if any(lc.lower() == col.lower() for lc in lon_cols):
            found_lon_col = col
        if found_lat_col and found_lon_col:
            break

    if found_lat_col and found_lon_col:
        # 緯度/経度が欠落しているか無効な行を除外します
        df = df.dropna(subset=[found_lat_col, found_lon_col])
        # 座標が数値であることを確認する
        df[found_lat_col] = pd.to_numeric(df[found_lat_col], errors='coerce')
        df[found_lon_col] = pd.to_numeric(df[found_lon_col], errors='coerce')
        df = df.dropna(subset=[found_lat_col, found_lon_col])

        if not df.empty:
            geometry = gpd.points_from_xy(df[found_lon_col], df[found_lat_col])
            gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
            print(f"CSV から正常にロードされ、{suffix_key_for_log}の GeoDataFrame に変換されました。")
            return gdf
        else:
            print(f"CSV は見つかりましたが、{suffix_key_for_log}をクリーニングした後、有効な座標データがありませんでした。")
            return None
    else:
        print(f"CSV は見つかりましたが、{suffix_key_for_log}の緯度({lat_cols})列と経度({lon_cols})列を識別できませんでした。")
        return None

# 破損したXML/GMLファイルから正規表現でデータを救出するフォールバックパース
def _parse_xml_fallback(xml_path):
    try:
        print(f"XMLファイルのテキスト正規表現によるサルベージ抽出を開始します: {xml_path}")
        with open(xml_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        # 各 EvacuationFacilities ノードのブロックを抽出する (ヌル文字を含まないよう [\s\S] パターンを使用)
        facilities = re.findall(r'<ksj:EvacuationFacilities[^>]*>([\s\S]*?)</ksj:EvacuationFacilities>', content)
        if not facilities:
            # prefixが省略されているか異なるケースの対応
            facilities = re.findall(r'<[^:>]*EvacuationFacilities[^>]*>([\s\S]*?)</[^:>]*EvacuationFacilities>', content)

        records = []
        for fac_xml in facilities:
            data = {}

            # タグと値を抽出するシンプルな正規表現
            tags = re.findall(r'<ksj:([^>]+)>([^<]+)</ksj:\1>', fac_xml)
            if not tags:
                tags = re.findall(r'<([^:>]+)>([^<]+)</\1>', fac_xml)

            for tag, val in tags:
                data[tag] = val.strip()

            # gml:pos 要素の抽出
            pos_match = re.search(r'<gml:pos[^>]*>([^<]+)</gml:pos>', fac_xml)
            if not pos_match:
                pos_match = re.search(r'<[^:>]*pos[^>]*>([^<]+)</[^:>]*pos>', fac_xml)

            if pos_match:
                coords = pos_match.group(1).strip().split()
                if len(coords) == 2:
                    data['lat'] = float(coords[0])
                    data['lon'] = float(coords[1])
                    records.append(data)

        if records:
            df = pd.DataFrame(records)
            geometry = gpd.points_from_xy(df['lon'], df['lat'])
            gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
            print(f"XMLの正規表現パースにより、破損ファイルから {len(gdf)} 件の避難施設データを正常にサルベージしました！")
            return gdf
    except Exception as e:
        print(f"XMLの正規表現サルベージ処理でエラーが発生しました: {e}")
    return None

# 圧縮ファイルから空間データファイルを検索し、GeoDataFrameとしてロードするヘルパー関数
def _find_and_load_spatial_data(temp_dir, suffix_key_for_log, csv_to_gdf_func):
    gdf = None
    found_file_path = None

    # 優先順位: GeoJSON, Shapefile, XML/GML, CSV
    for root, dirs, files in os.walk(temp_dir):
        for file in files:
            if file.lower().endswith('.geojson'):
                found_file_path = os.path.join(root, file)
                try:
                    gdf = gpd.read_file(found_file_path)
                    print(f"読み込まれたGeoJSON: {found_file_path}")
                    return gdf
                except Exception as e:
                    print(f"GeoJSON '{found_file_path}'の読み取りエラー: {e}")
                    found_file_path = None

    if gdf is None: # Shapefileの探索
         for root, dirs, files in os.walk(temp_dir):
            for file in files:
                if file.lower().endswith('.shp'):
                    found_file_path = os.path.join(root, file)
                    try:
                        gdf = gpd.read_file(found_file_path)
                        print(f"読み込まれたシェープファイル: {found_file_path}")
                        return gdf
                    except Exception as e:
                        print(f"シェープファイル '{found_file_path}'の読み取り・パースエラー (破損の可能性あり): {e}")
                        found_file_path = None

    if gdf is None: # XML/GMLの探索 (Shapefile破損時の強力なフォールバック)
         for root, dirs, files in os.walk(temp_dir):
            for file in files:
                if file.lower().endswith('.xml') and not file.startswith('KS-META-'):
                    found_file_path = os.path.join(root, file)
                    gdf = _parse_xml_fallback(found_file_path)
                    if gdf is not None:
                        return gdf

    if gdf is None: # CSVを検索
        for root, dirs, files in os.walk(temp_dir):
            for file in files:
                if file.lower().endswith('.csv'):
                    found_file_path = os.path.join(root, file)
                    try:
                        df = pd.read_csv(found_file_path)
                        gdf = csv_to_gdf_func(df, suffix_key_for_log)
                        if gdf is not None:
                            print(f"読み込まれたCSV: {found_file_path} (GeoDataFrameに変換済み)")
                        return gdf
                    except Exception as e:
                        print(f"CSV '{found_file_path}'の読み取りエラー: {e}")
                        found_file_path = None

    if gdf is None:
        print(f"一時ディレクトリ '{temp_dir}'に有効なジオデータファイルが見つかりません。")
    return gdf

# ZIPファイルの展開を試み、失敗した場合はOSのunzipコマンドでフォールバックして展開する関数
def _extract_zip_with_fallback(zip_bytes, target_dir):
    try:
        # 1. 通常のPython zipfileモジュールでの展開を試行
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            zf.extractall(target_dir)
            return True
    except Exception as py_err:
        print(f"[情報] Pythonのzipfileによる展開でエラーが発生しました。zip -FF による修復を伴うOSの展開を試みます: {py_err}")
        try:
            corrupted_zip = os.path.join(target_dir, "_corrupted.zip")
            fixed_zip = os.path.join(target_dir, "_fixed.zip")
            with open(corrupted_zip, "wb") as f:
                f.write(zip_bytes)

            # zip -FF コマンドを使って破損アーカイブの修復サルベージを試みる
            subprocess.run(["zip", "-FF", corrupted_zip, "--out", fixed_zip], capture_output=True, check=False)

            if os.path.exists(fixed_zip):
                # 修復後のZIPをターゲットフォルダに展開する
                subprocess.run(["unzip", "-o", fixed_zip, "-d", target_dir], capture_output=True, check=False)
                # 一時中間ファイルのクリーンアップ
                os.remove(corrupted_zip)
                os.remove(fixed_zip)
                return True
            else:
                # 修復失敗した場合はそのまま通常のunzipによる強制展開を代替試行
                subprocess.run(["unzip", "-o", corrupted_zip, "-d", target_dir], capture_output=True, check=False)
                os.remove(corrupted_zip)
                return True
        except Exception as os_err:
            print(f"[エラー] OS of unzipコマンドおよび修復展開も失敗しました: {os_err}")
            return False

# 国土地理院 GeoJSON/Shapefile/CSV データをダウンロードして処理するヘルパー関数
def _get_gsi_data(prefix, suffix_key, info, base_url, csv_to_gdf_func):
    zip_filename = f'{prefix}{suffix_key}.zip'
    zip_url = f'{base_url}{zip_filename}'
    gdf = None
    print(f"ダウンロードと処理中: {zip_url}")
    try:
        response = requests.get(zip_url)
        response.raise_for_status()
        temp_dir = tempfile.mkdtemp()
        try:
            if _extract_zip_with_fallback(response.content, temp_dir):
                gdf = _find_and_load_spatial_data(temp_dir, suffix_key, csv_to_gdf_func)
        finally:
            if temp_dir and os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
    except requests.exceptions.RequestException as e:
        print(f"{zip_url}のダウンロード中にエラーが発生しました: {e}")
    except Exception as e:
        print(f"{zip_filename}の処理中に予期しないエラーが発生しました: {e}")
    return gdf

# 国土数値情報 GeoJSON/Shapefile/CSV データをダウンロードして処理するためのヘルパー関数
def _get_ksj_data(prefix, base_url, csv_to_gdf_func):
    zip_filename = f'{prefix}_GML.zip'
    zip_url = f'{base_url}/{zip_filename}'
    gdf = None
    print(f"ダウンロードと処理中: {zip_url}")
    try:
        response = requests.get(zip_url)
        response.raise_for_status()
        temp_dir = tempfile.mkdtemp()
        try:
            if _extract_zip_with_fallback(response.content, temp_dir):
                gdf = _find_and_load_spatial_data(temp_dir, prefix, csv_to_gdf_func)
        finally:
            if temp_dir and os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
    except requests.exceptions.RequestException as e:
        print(f"{zip_url}のダウンロード中にエラーが発生しました: {e}")
    except Exception as e:
        print(f"{zip_filename}の処理中に予期しないエラーが発生しました: {e}")
    return gdf

# 避難所等データのジオデータフレームを作成する
base_url = GSI_URL
prefixes = PREF_CODE
suffixes = {'_1': {'name': '指定避難所', 'fields': ['共通ID','施設・場所名','住所','指定緊急避難場所との住所同一','その他市町村長が必要と認める事項','受入対象者','備考']},
            '_2': {'name': '指定緊急避難場所', 'fields': ['共通ID','施設・場所名','住所','洪水','崖崩れ、土石流及び地滑り','高潮','地震','津波','大規模な火事','内水氾濫','火山現象','指定避難所との住所同一','備考']}}

shelter_gdfs_list = []
evac_site_gdfs_list = []

for prefix in prefixes:
    for suffix_key, info in suffixes.items():
        gdf = _get_gsi_data(prefix, suffix_key, info, base_url, csv_to_gdf)
        if gdf is not None:
            if suffix_key == '_1':
                shelter_gdfs_list.append(gdf)
            elif suffix_key == '_2':
                evac_site_gdfs_list.append(gdf)
        else:
            print(f"{prefix}{suffix_key}の空間データをロードできませんでした。")

# タイプごとに GeoDataFrame を連結する
if shelter_gdfs_list:
    combined_shelters_gdf = gpd.pd.concat(shelter_gdfs_list, ignore_index=True)
    cols_to_replace_shelters = ['指定緊急避難場所との住所同一']
    combined_shelters_gdf = _replace_categorical_values(combined_shelters_gdf, cols_to_replace_shelters)

    folium.GeoJson(
        combined_shelters_gdf,
        attribution='&copy; <a href="https://hinanmap.gsi.go.jp/hinanjocp/hinanbasho/koukaidate.html" target="_blank" rel="noopener">国土地理院</a> ',
        name=suffixes['_1']['name'],
        tooltip=folium.GeoJsonTooltip(fields=suffixes['_1']['fields']),
        popup=folium.GeoJsonPopup(fields=suffixes['_1']['fields']),
        zoom_on_click=True,
    ).add_to(fmap1)

if evac_site_gdfs_list:
    combined_evac_sites_gdf = gpd.pd.concat(evac_site_gdfs_list, ignore_index=True)
    cols_to_replace_evac_sites = ['洪水', '崖崩れ、土石流及び地滑り', '高潮', '地震', '津波', '大規模な火事', '内水氾濫', '火山現象', '指定避難所との住所同一']
    combined_evac_sites_gdf = _replace_categorical_values(combined_evac_sites_gdf, cols_to_replace_evac_sites)

    folium.GeoJson(
        combined_evac_sites_gdf,
        attribution='&copy; <a href="https://hinanmap.gsi.go.jp/hinanjocp/hinanbasho/koukaidate.html" target="_blank" rel="noopener">国土地理院</a> ',
        name=suffixes['_2']['name'],
        tooltip=folium.GeoJsonTooltip(fields=suffixes['_2']['fields']),
        popup=folium.GeoJsonPopup(fields=suffixes['_2']['fields']),
        zoom_on_click=True,
    ).add_to(fmap1)

# 避難施設データ2012年度版の読み込み
base_url2 = KSJ_URL
prefixes2 = P20_ZIP_NAME
shelter_2012_gdfs_list = []
combined_shelters_2012_gdf = []

for prefix in prefixes2:
    gdf = _get_ksj_data(prefix, base_url2, csv_to_gdf)
    if gdf is not None:
        shelter_2012_gdfs_list.append(gdf)
    else:
        print(f"{prefix}の空間データをロードできませんでした。")

if shelter_2012_gdfs_list:
    combined_shelters_2012_gdf = gpd.pd.concat(shelter_2012_gdfs_list, ignore_index=True)

    # カラムのリネーム定義
    rename_dict = {
        'P20_001':'行政区域コード','P20_002':'名称','P20_003':'住所','P20_004':'施設の種類',
        'P20_005':'収容人数','P20_006':'施設規模(m2)','P20_007':'地震災害','P20_008':'津波災害',
        'P20_009':'水害','P20_010':'火山災害','P20_011':'災害分類その他','P20_012':'災害分類指定なし'
    }

    # XML直接パース時はタグ名（小文字やprefixなしなど）に合わせるフォールバックを考慮する
    actual_rename = {k: v for k, v in rename_dict.items() if k in combined_shelters_2012_gdf.columns}
    combined_shelters_2012_gdf = combined_shelters_2012_gdf.rename(columns=actual_rename)

    cols_to_replace_shelters_2012 = ['地震災害', '津波災害', '水害', '火山災害', '災害分類その他', '災害分類指定なし']
    combined_shelters_2012_gdf = _replace_categorical_values(combined_shelters_2012_gdf, cols_to_replace_shelters_2012)

    display_fields_for_map = ['行政区域コード','名称', '住所', '施設の種類','収容人数','施設規模(m2)','地震災害', '津波災害', '水害', '火山災害', '災害分類その他', '災害分類指定なし']
    tooltip_fields = [field for field in display_fields_for_map if field in combined_shelters_2012_gdf.columns]
    popup_fields = [field for field in display_fields_for_map if field in combined_shelters_2012_gdf.columns]

    if not tooltip_fields:
        non_geometry_cols = [col for col in combined_shelters_2012_gdf.columns if col != combined_shelters_2012_gdf.geometry.name]
        tooltip_fields = non_geometry_cols[:3]
        popup_fields = non_geometry_cols[:3]

    tooltip_fields = list(dict.fromkeys(tooltip_fields))
    popup_fields = list(dict.fromkeys(popup_fields))

    folium.GeoJson(
        combined_shelters_2012_gdf,
        attribution='&copy; <a href="https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-P20.html" target="_blank" rel="noopener">国土数値情報</a> ',
        name='避難施設データ 2012年度（平成24年度）版',
        tooltip=folium.GeoJsonTooltip(fields=tooltip_fields),
        popup=folium.GeoJsonPopup(fields=popup_fields),
        zoom_on_click=True,
        show=False,
        marker=folium.CircleMarker(radius=3, fill=True, color='blue', fill_color='blue', fill_opacity=0.6)
    ).add_to(fmap1)
else:
    print("'combined_shelters_2012_gdf'は作成されませんでした。")

# 統計情報の表示
if 'combined_shelters_gdf' in locals():
    num_shelters = len(combined_shelters_gdf)
    print(f"指定避難所(combined_shelters_gdf)の総数は、{num_shelters}箇所。")
if 'combined_evac_sites_gdf' in locals():
    num_evac_sites = len(combined_evac_sites_gdf)
    print(f"指定緊急避難場所(combined_evac_sites_gdf)の総数は、{num_evac_sites}箇所。")
if isinstance(combined_shelters_2012_gdf, gpd.GeoDataFrame) and not combined_shelters_2012_gdf.empty:
    num_shelters_2012 = len(combined_shelters_2012_gdf)
    print(f"避難施設2012年度版(combined_shelters_2012_gdf)の総数は、{num_shelters_2012}箇所。")

# 各GDFの中身の表示
if 'combined_shelters_gdf' in locals(): display(combined_shelters_gdf.head(3))
if 'combined_evac_sites_gdf' in locals(): display(combined_evac_sites_gdf.head(3))
if isinstance(combined_shelters_2012_gdf, gpd.GeoDataFrame) and not combined_shelters_2012_gdf.empty:
    display(combined_shelters_2012_gdf.head(3))

# ベースマップの追加
base_maps = [
    ('https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png', '地理院地図', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> '),
    ('https://cyberjapandata.gsi.go.jp/xyz/english/{z}/{x}/{y}.png', 'English', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> '),
    ('http://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', 'Esri Satellite', 'Esri'),
    ('https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.png', '全国最新写真（シームレス）', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> ')
]
for tiles, name, attr in base_maps:
    folium.raster_layers.TileLayer(tiles=tiles, attr=attr, name=name, overlay=False, control=True).add_to(fmap1)

# 災害ハザードレイヤーの追加
disaster_layers = {
    '01_flood_l2_shinsuishin_kuni_data': '洪水浸水想定区域（想定最大規模）',
    '01_flood_l2_keizoku_data': '浸水継続時間（想定最大規模）',
    '01_flood_l2_kaokutoukai_hanran_data': '家屋倒壊等氾濫想定区域（氾濫流）',
    '01_flood_l2_kagan_data': '家屋倒壊等氾濫想定区域（河岸侵食）',
    '02_naisui_data': '内水（雨水出水）浸水想定区域',
    '03_hightide_l2_shinsuishin_data': '高潮浸水想定区域',
    '04_tsunami_newlegend_data': '津波浸水想定',
    '05_dosekiryukeikaikuiki': '土砂災害警戒区域（土石流）',
    '05_kyukeishakeikaikuiki': '土砂災害警戒区域（急傾斜地の崩壊）',
    '05_jisuberikeikaikuiki': '土砂災害警戒区域（地すべり）',
    '05_nadarekikenkasyo': '雪崩危険箇所'
}
for url_part, name in disaster_layers.items():
    folium.raster_layers.TileLayer(
        tiles=f'https://disaportal.gsi.go.jp/raster/{url_part}/{{z}}/{{x}}/{{y}}.png',
        fmt='image/png',
        attr='&copy; <a href="https://disaportal.gsi.go.jp/hazardmap/copyright/opendata.html" target="_blank" rel="noopener">ハザードマップポータルサイト</a> ',
        name=name, tms=False, overlay=True, control=True, opacity=0.7
    ).add_to(fmap1)

folium.LayerControl().add_to(fmap1)
plugins.Fullscreen(position="topright", title="Expand me", title_cancel="Exit me", force_separate_button=True).add_to(fmap1)
plugins.LocateControl(auto_start=False, flyToNextView=True, strings={"title": "現在地を表示"}).add_to(fmap1)

# 出力保存とダウンロード
filename = f"evacuation_map_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.html"
fmap1.save(filename)
try:
    files.download(filename)
except Exception as e:
    print(f"自動ダウンロードに失敗しました (環境依存): {e}")


地図の中央に配置する住所を英語またはローマ字で入力して下さい。
（例:「Tokyo Station」）また空白の場合はデフォルトの座標を使用します。
決定後Enterキーを押して下さい：mt sakurajima
地図の中心: 桜島, 鹿児島市, 鹿児島県, 892-8677, 日本 (31.5885088, 130.6659596) 
ダウンロードと処理中: https://japanhazardmap.github.io/GISDataOriginalZips/MLITGSI/DesignatedEvacuationSitesandShelters/45000_1.zip
読み込まれたGeoJSON: /tmp/tmpi2bkshth/45000_1.geojson
ダウンロードと処理中: https://japanhazardmap.github.io/GISDataOriginalZips/MLITGSI/DesignatedEvacuationSitesandShelters/45000_2.zip
読み込まれたGeoJSON: /tmp/tmp6xmcafs8/45000_2.geojson
ダウンロードと処理中: https://japanhazardmap.github.io/GISDataOriginalZips/MLITGSI/DesignatedEvacuationSitesandShelters/46000_1.zip
読み込まれたGeoJSON: /tmp/tmpinrzakox/46000_1.geojson
ダウンロードと処理中: https://japanhazardmap.github.io/GISDataOriginalZips/MLITGSI/DesignatedEvacuationSitesandShelters/46000_2.zip
読み込まれたGeoJSON: /tmp/tmptccmktmm/46000_2.geojson
ダウンロードと処理中: https://japanhazardmap.github.io/GISDataOriginalZips/MLITKSJ/P20/P20-12_45_GML.zip
読み込まれたシェープファイル: /tmp/tmphqqqjb6e/P20-12_45.shp
ダウンロードと処

,NO,共通ID,施設・場所名,住所,指定緊急避難場所との住所同一,その他市町村長が必要と認める事項,受入対象者,備考,geometry
0,1,E4520900001111,岡元小学校体育館,宮崎県えびの市大字浦371,×非該当×,None,None,,POINT (130.7857 32.02126)
1,2,E4520900002111,えびの市国際交流センター,宮崎県えびの市大字榎田388-1,×非該当×,None,None,,POINT (130.81088 32.05663)
2,3,E4520900023112,南岡松公民館,宮崎県えびの市大字岡松1412-2,該当,None,None,,POINT (130.75419 32.04685)


,NO,共通ID,施設・場所名,住所,洪水,崖崩れ、土石流及び地滑り,高潮,地震,津波,大規模な火事,内水氾濫,火山現象,指定避難所との住所同一,備考,geometry
0,1,E4520900023201,南岡松公民館,宮崎県えびの市大字岡松1412-2,該当,該当,×非該当×,×非該当×,×非該当×,×非該当×,該当,×非該当×,該当,,POINT (130.75419 32.04685)
1,2,E4520900009201,飯野地区コミュニティセンター,宮崎県えびの市大字原田112-11,該当,該当,×非該当×,×非該当×,×非該当×,×非該当×,該当,該当,該当,,POINT (130.8668 32.04654)
2,3,E4520900011201,飯野駅前地区体育館,宮崎県えびの市大字原田2176,該当,該当,×非該当×,該当,×非該当×,×非該当×,該当,該当,該当,,POINT (130.8728 32.03357)


,行政区域コード,名称,住所,施設の種類,収容人数,施設規模(m2),地震災害,津波災害,水害,火山災害,災害分類その他,災害分類指定なし,レベル,備考,緯度,経度,NO,geometry
0,45201,（有）加藤えのき事務所,宮崎県宮崎市高岡町浦之名2945-2,避難施設,50,108,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,3,None,31.951249,131.271001,1,POINT (131.271 31.95125)
1,45201,ＧａｒｄｅｎＣｉｔｙヴァンサンカン,宮崎県宮崎市佐土原町下那珂695-8,津波避難所,-1,-1,×非該当×,該当,×非該当×,×非該当×,×非該当×,×非該当×,3,None,32.007749,131.479141,2,POINT (131.47914 32.00775)
2,45201,ＪＡ営農センター,宮崎県宮崎市清武町加納字須木田885,避難施設,200,340,×非該当×,×非該当×,×非該当×,×非該当×,×非該当×,該当,3,None,31.856746,131.408715,3,POINT (131.40872 31.85675)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#完成した地図fmap1を表示させる
fmap1